In [165]:
import os
import pandas as pd
import numpy as np
import random

from tqdm import tqdm
from datasets import load_dataset, Dataset, Image, List, Value, Features
from dotenv import load_dotenv
from huggingface_hub import login

tqdm.pandas()
load_dotenv(dotenv_path="../../.env")
login(os.getenv("HF_API_KEY"))

In [67]:
dataset = load_dataset("laicsiifes/flickr30k-pt-br-5k")

In [220]:
df_data = dataset["test"].to_pandas()

In [221]:
def select_incorrect_data(row, incorrect_sample_size, incorrect_data, replacement=False, reproducible=True):
    random_state = row['img_id'] if reproducible else None
    current_filename = row['filename']
    incorrect_sample = incorrect_data.loc[~incorrect_data.filename.isin([current_filename])].sample(n=incorrect_sample_size, replace=replacement, random_state=random_state)
    row['incorrect_group_filenames'] = incorrect_sample['filename'].values
    row['incorrect_group_sentids'] = incorrect_sample['sentids'].values
    row['incorrect_group'] = incorrect_sample['caption'].values
    return row

def generate_grouped_dataset(dataset, correct_sample_size, incorrect_sample_size, reproducible=True):
    df = dataset.to_pandas()
    df['correct_group_sentids'] = df['sentids'].progress_apply(lambda x: random.sample(x.tolist(), correct_sample_size))
    df['correct_group'] = df.progress_apply(
        lambda x: [
            x['caption'].tolist()[i] for i in range(len(x['caption'])) if i in [int(sentid) % len(x['caption']) for sentid in x['correct_group_sentids']]
        ],
        axis=1
    )
    df['control_group_sentids'] = df.progress_apply(lambda x: [i for i in x['sentids'].tolist() if i not in x['correct_group_sentids']], axis=1)
    df['control_group'] = df.progress_apply(
        lambda x: [
            x['caption'].tolist()[i] for i in range(len(x['caption'])) if i in [int(sentid) % len(x['caption']) for sentid in x['control_group_sentids']]
        ],
        axis=1
    )
    
    incorrect_data = df.explode(['caption', 'sentids'])
    
    df = df.progress_apply(
        lambda row: select_incorrect_data(
            row=row,
            incorrect_sample_size=incorrect_sample_size,
            incorrect_data=incorrect_data,
            replacement=False
        ),
        axis=1
    )
    
    features = Features({
        'image': Image(mode=None, decode=True),
        'caption': List(Value('string')),
        'sentids': List(Value('int32')),
        'split': Value('string'),
        'img_id': Value('string'),
        'filename': Value('string'),
        'correct_group_sentids': List(Value('int32')),
        'correct_group': List(Value('string')),
        'control_group_sentids': List(Value('int32')),
        'control_group': List(Value('string')),
        'incorrect_group_filenames': List(Value('string')),
        'incorrect_group_sentids': List(Value('int32')),
        'incorrect_group': List(Value('string'))
    })
    
    return Dataset.from_pandas(df, features=features)

In [222]:
df = generate_grouped_dataset(df_data, correct_sample_size=2, incorrect_sample_size=10)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:01<00:00, 562.40it/s]


In [225]:
df.to_pandas()

,image,caption,sentids,split,img_id,filename,correct_group_sentids,correct_group,control_group_sentids,control_group,incorrect_group_filenames,incorrect_group_sentids,incorrect_group
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,[O homem com orelhas furadas usa óculos e chap...,"[125, 126, 127, 128, 129]",test,25,1007129816.jpg,"[127, 129]",[Um homem com medidores e óculos está usando u...,"[125, 126, 128]",[O homem com orelhas furadas usa óculos e chap...,"[5026046208.jpg, 6897514777.jpg, 2860040276.jp...","[126581, 143027, 44911, 42136, 107490, 71302, ...","[Um homem e uma mulher trabalham no campo, amb..."
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,[Um cachorro preto e branco corre em um jardim...,"[170, 171, 172, 173, 174]",test,34,1009434119.jpg,"[174, 172]","[Um cachorro preto e branco corre pela grama.,...","[170, 171, 173]",[Um cachorro preto e branco corre em um jardim...,"[4736841029.jpg, 4460747081.jpg, 2844641033.jp...","[110815, 96772, 44351, 28621, 92098, 93615, 27...",[Uma criança com cabelo comprido e camisa rosa...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,[Uma jovem estudante dando um chute para baixo...,"[255, 256, 257, 258, 259]",test,51,101362133.jpg,"[258, 259]",[Uma garota com uniforme de caratê quebrando u...,"[255, 256, 257]",[Uma jovem estudante dando um chute para baixo...,"[2696866120.jpg, 327142149.jpg, 4923272678.jpg...","[38753, 61527, 122129, 114178, 62713, 36319, 9...","[Um cachorro rola no chão., Um chão de terra é..."
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,"[Cinco pilotos de motos de neve, todos usando ...","[445, 446, 447, 448, 449]",test,89,102617084.jpg,"[446, 447]",[Cinco pessoas vestindo jaquetas de inverno e ...,"[445, 448, 449]","[Cinco pilotos de motos de neve, todos usando ...","[5163992452.jpg, 6775387932.jpg, 4549977232.jp...","[128131, 141644, 100519, 132037, 24512, 37538,...",[Duas mulheres africanas vestindo saias roxas ...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,[Dois homens sentados no telhado de uma casa e...,"[485, 486, 487, 488, 489]",test,97,10287332.jpg,"[488, 489]",[As pessoas estão consertando o telhado de uma...,"[485, 486, 487]",[Dois homens sentados no telhado de uma casa e...,"[2217728745.jpg, 4732745499.jpg, 7348289414.jp...","[20209, 110577, 146903, 106114, 74870, 67584, ...","[Um homem em um bar., Uma mulher mais velha es..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,[Os corredores de maratona estão correndo em u...,"[153905, 153906, 153907, 153908, 153909]",test,30781,900144365.jpg,"[153909, 153906]",[Duas mulheres corredoras de shorts esportivos...,"[153905, 153907, 153908]",[Os corredores de maratona estão correndo em u...,"[2582413611.jpg, 101362133.jpg, 8220955383.jpg...","[34609, 258, 152472, 123124, 61226, 47897, 122...",[Adolescentes se saindo melhor em uma foto de ...
996,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,[Uma mulher oriental de chapéu andando de bici...,"[154215, 154216, 154217, 154218, 154219]",test,30843,94024624.jpg,"[154215, 154217]",[Uma mulher oriental de chapéu andando de bici...,"[154216, 154218, 154219]",[Uma mulher asiática usando um chapéu tradicio...,"[3072673694.jpg, 1039637574.jpg, 2208662604.jp...","[53145, 620, 19860, 128134, 45370, 140212, 772...",[Um grupo de jovens está encostado em uma cerc...
997,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,"[Meninos estão na calçada perto das árvores, e...","[154465, 154466, 154467, 154468, 154469]",test,30893,95758790.jpg,"[154467, 154468]",[Uma criança observa outra brincadeira no chão...,"[154465, 154466, 154469]","[Meninos estão na calçada perto das árvores, e...","[1489286545.jpg, 388837010.jpg, 3449846784.jpg...","[8618, 84444, 68927, 37749, 42598, 76794, 1332...",[Um homem e seu cachorro observam o pôr do sol...
998,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,"[Um homem, que está vestindo um casaco bege, e...","[154710, 154711, 154712, 154713, 15471

---

# Observing the results

---

In [13]:
import pandas as pd
import os

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [14]:
df = pd.read_csv('../results/flickr30k_pt_human_generated/2_vs_2/results.csv')

In [18]:
df.drop(columns=['caption']).head(1)

,image,sentids,split,img_id,filename,correct_group_sentids,correct_group,control_group_sentids,control_group,incorrect_group_filenames,incorrect_group_sentids,incorrect_group,bertscore_precision_correct,bertscore_recall_correct,bertscore_f1_correct,bertscore_hashcode_correct,clipscore_correct,ref_clipscore_correct,rouge1_correct,rouge2_correct,rougeL_correct,rougeLsum_correct,meteor_correct,bleu_correct,precisions_correct,brevity_penalty_correct,length_ratio_correct,translation_length_correct,reference_length_correct,bertscore_precision_incorrect,bertscore_recall_incorrect,bertscore_f1_incorrect,bertscore_hashcode_incorrect,clipscore_incorrect,ref_clipscore_incorrect,rouge1_incorrect,rouge2_incorrect,rougeL_incorrect,rougeLsum_incorrect,meteor_incorrect,bleu_incorrect,precisions_incorrect,brevity_penalty_incorrect,length_ratio_incorrect,translation_length_incorrect,reference_length_incorrect
0,<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x461 at 0x77C7D2188B30>,"[1, 2, 3, 4, 5]",test,25,1007129816.jpg,"[3, 5]","['Um homem de óculos vestindo um chapéu laranja, feito de tricô.', 'Um homem de óculos e chapéu laranja bordado com embalagens de cerveja.']","[1, 2, 4]","['Um homem branco de óculos e costeletas está de perfil enquanto usa uma touca de crochê com propaganda de cerveja está em um ambiente iluminado com outras pessoas ao fundo.', 'Um homem usando um chapéu de crochet laranja, óculos pretos e um alargador de orelha caminha por um corredor.', 'Um homem com um chapéu laranja e óculos usando uma blusa cinza.']","['101362133.jpg', '1009434119.jpg']","[2, 1]","['Homem com roupa de taekwondo segura pedaço de madeira para uma menina chutar.', 'Um cachorro usando uma coleira verde corre em um quintal gramado cercado por uma cerca branca de madeira.']","[0.7329137325286865, 0.6968921422958374]","[0.7973186373710632, 0.7432618141174316]","[0.7637608051300049, 0.6989191770553589]",neuralmind/bert-base-portuguese-cased_L12_no-idf_version=0.3.12(hug_trans=4.46.1),"[0.8677861839532852, 0.9124748408794403]","[1.2394010653194005, 1.2401243515596718]","[0.5333333333333333, 0.5833333333333334]","[0.1904761904761905, 0.18181818181818182]","[0.39999999999999997, 0.3870967741935484]","[0.39999999999999997, 0.3870967741935484]","[0.48822605965463106, 0.4092548076923077]","[0.0, 0.0]","['[0.7272727272727273, 0.3, 0.0, 0.0]', '[0.75, 0.36363636363636365, 0.1, 0.0]']","[0.9131007162822622, 1.0]","[0.9166666666666666, 1.0]","[11, 12]","[12, 12]","[0.47723573446273804, 0.5880365967750549]","[0.5572360754013062, 0.6129875183105469]","[0.4888671636581421, 0.5824325680732727]",neuralmind/bert-base-portuguese-cased_L12_no-idf_version=0.3.12(hug_trans=4.46.1),"[0.12400043196976185, 0.0]","[0.22430953643100604, 0.0]","[0.18604651162790697, 0.27027027027027023]","[0.0, 0.07142857142857142]","[0.16, 0.21621621621621623]","[0.16, 0.21621621621621623]","[0.24093511450381686, 0.18382352941176472]","[0.0, 0.0]","['[0.3076923076923077, 0.0, 0.0, 0.0]', '[0.3888888888888889, 0.11764705882352941, 0.0, 0.0]']","[1.0, 1.0]","[1.0833333333333333, 1.5]","[13, 18]","[12, 12]"
